In [3]:
import pandas as pd

In [16]:
newleg = pd.read_csv("./new1.csv")
newphis = pd.read_csv("./phishing_urls.csv")
print(newleg.count(),"\n" ,newphis.count())

url      161259
label    161258
dtype: int64 
 URL      134850
label    134850
dtype: int64


In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import tldextract
import re
import pickle

# Load the datasets
phishing_df = pd.read_csv('./phishing_urls.csv')
legitimate_df = pd.read_csv('./new1.csv')

# Rename columns for consistency
phishing_df.columns = ['url', 'label']
legitimate_df.columns = ['url', 'label']

# Clean the datasets
phishing_df = phishing_df.dropna(subset=['url', 'label'])
legitimate_df = legitimate_df.dropna(subset=['url', 'label'])

# Ensure labels are consistent (1 for phishing, 0 for legitimate)
phishing_df['label'] = phishing_df['label'].astype(int)
legitimate_df['label'] = legitimate_df['label'].astype(float).astype(int)

# Combine the datasets
combined_df = pd.concat([phishing_df, legitimate_df], ignore_index=True)
print(combined_df.count())

# Feature extraction functions
def extract_features(url):
    try:
        # Parse URL components
        parsed = tldextract.extract(url)
        domain = parsed.domain
        tld = parsed.suffix
        subdomain = parsed.subdomain
        hostname = f"{subdomain}.{domain}.{tld}" if subdomain else f"{domain}.{tld}"

        # Initialize features
        features = {}

        # 1. URL length
        features['url_length'] = len(url)

        # 2. Domain length
        features['domain_length'] = len(hostname)

        # 3. Number of dots in URL
        features['num_dots'] = url.count('.')

        # 4. Number of hyphens in URL
        features['num_hyphens'] = url.count('-')

        # 5. Presence of suspicious keywords in URL
        suspicious_keywords = ['login', 'verify', 'account', 'secure', 'update', 'bank', 'paypal', 'signin']
        features['has_suspicious_keyword'] = int(any(keyword in url.lower() for keyword in suspicious_keywords))

        # 6. Number of subdomains
        features['num_subdomains'] = len(subdomain.split('.')) if subdomain else 0

        # 7. Presence of HTTPS
        features['has_https'] = int(url.lower().startswith('https'))

        # 8. Presence of digits in domain
        features['has_digits_in_domain'] = int(any(char.isdigit() for char in domain))

        # 9. TLD (encoded later)
        features['tld'] = tld if tld else 'unknown'

        # 10. Path length
        path = url.split(hostname)[-1]
        features['path_length'] = len(path)

        # 11. Number of query parameters
        query = url.split('?')[-1] if '?' in url else ''
        features['num_query_params'] = len(query.split('&')) if query else 0

        # 12. Presence of IP address in URL
        ip_pattern = r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b'
        features['has_ip_address'] = int(bool(re.search(ip_pattern, url)))

        return features
    except Exception as e:
        print(f"Error processing URL {url}: {e}")
        return None

# Extract features for all URLs
feature_list = []
labels = []
skipped = 0
processed = 0

for index, row in combined_df.iterrows():
    url = row['url']
    label = row['label']
    features = extract_features(url)
    if features:
        feature_list.append(features)
        labels.append(label)
        processed += 1
    else:
        skipped += 1

print(f"\n✅ Total URLs processed: {processed}")
print(f"⚠️ Total URLs skipped due to errors: {skipped}")


# Convert features to DataFrame
features_df = pd.DataFrame(feature_list)

# Encode categorical feature 'tld'
label_encoder = LabelEncoder()
features_df['tld'] = label_encoder.fit_transform(features_df['tld'])

# Convert labels to numpy array
labels = np.array(labels)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features_df, labels, test_size=0.2, random_state=42, stratify=labels)

# Initialize and train Random Forest Classifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_classifier.fit(X_train, y_train)

# Make predictions
y_pred = rf_classifier.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Phishing']))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_classifier.feature_importances_
}).sort_values(by='importance', ascending=False)
print("\nFeature Importance:")
print(feature_importance)

# Save the model and label encoder
with open('rf_phishing_model.pkl', 'wb') as f:
    pickle.dump(rf_classifier, f)

with open('tld_label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

print("\nModel and label encoder saved successfully.")

url      296108
label    296108
dtype: int64

✅ Total URLs processed: 296108
⚠️ Total URLs skipped due to errors: 0
Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     32252
    Phishing       1.00      1.00      1.00     26970

    accuracy                           1.00     59222
   macro avg       1.00      1.00      1.00     59222
weighted avg       1.00      1.00      1.00     59222


Confusion Matrix:
[[32252     0]
 [    0 26970]]

Feature Importance:
                   feature  importance
5           num_subdomains    0.336280
6                has_https    0.324482
2                 num_dots    0.198671
0               url_length    0.103979
1            domain_length    0.030582
9              path_length    0.002593
8                      tld    0.002404
7     has_digits_in_domain    0.000502
3              num_hyphens    0.000368
10        num_query_params    0.000113
4   has_suspicio

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import tldextract
import re
import pickle

# Load the trained model and label encoder
with open('rf_phishing_model.pkl', 'rb') as f:
    rf_classifier = pickle.load(f)

with open('tld_label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

# List of custom URLs to predict
custom_urls = [
    "https://www.saffronart.com",
    "https://1.bayarefastrlz.top/"
]

# Feature extraction function (same as used during training)
def extract_features(url):
    try:
        # Parse URL components
        parsed = tldextract.extract(url)
        domain = parsed.domain
        tld = parsed.suffix
        subdomain = parsed.subdomain
        hostname = f"{subdomain}.{domain}.{tld}" if subdomain else f"{domain}.{tld}"

        # Initialize features
        features = {}

        # 1. URL length
        features['url_length'] = len(url)

        # 2. Domain length
        features['domain_length'] = len(hostname)

        # 3. Number of dots in URL
        features['num_dots'] = url.count('.')

        # 4. Number of hyphens in URL
        features['num_hyphens'] = url.count('-')

        # 5. Presence of suspicious keywords in URL
        suspicious_keywords = ['login', 'verify', 'account', 'secure', 'update', 'bank', 'paypal', 'signin']
        features['has_suspicious_keyword'] = int(any(keyword in url.lower() for keyword in suspicious_keywords))

        # 6. Number of subdomains
        features['num_subdomains'] = len(subdomain.split('.')) if subdomain else 0

        # 7. Presence of HTTPS
        features['has_https'] = int(url.lower().startswith('https'))

        # 8. Presence of digits in domain
        features['has_digits_in_domain'] = int(any(char.isdigit() for char in domain))

        # 9. TLD (to be encoded)
        features['tld'] = tld if tld else 'unknown'

        # 10. Path length
        path = url.split(hostname)[-1]
        features['path_length'] = len(path)

        # 11. Number of query parameters
        query = url.split('?')[-1] if '?' in url else ''
        features['num_query_params'] = len(query.split('&')) if query else 0

        # 12. Presence of IP address in URL
        ip_pattern = r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b'
        features['has_ip_address'] = int(bool(re.search(ip_pattern, url)))

        return features
    except Exception as e:
        print(f"Error processing URL {url}: {e}")
        return None

# Extract features for custom URLs
feature_list = []
valid_urls = []

for url in custom_urls:
    features = extract_features(url)
    if features:
        feature_list.append(features)
        valid_urls.append(url)

# Convert features to DataFrame
features_df = pd.DataFrame(feature_list)

# Encode TLD using the loaded label encoder
# Handle unseen TLDs by mapping to the encoded value of 'com'
default_tld = 'com'  # Default TLD to use for unseen TLDs
default_tld_encoded = label_encoder.transform([default_tld])[0]  # Get encoded value of 'com'
features_df['tld'] = features_df['tld'].apply(
    lambda x: x if x in label_encoder.classes_ else default_tld
)
features_df['tld'] = label_encoder.transform(features_df['tld'])

# Make predictions
predictions = rf_classifier.predict(features_df)
probabilities = rf_classifier.predict_proba(features_df)

# Output results
print("\nPredictions for Custom URLs:")
print("-" * 50)
for url, pred, prob in zip(valid_urls, predictions, probabilities):
    label = "Phishing" if pred == 1 else "Legitimate"
    prob_phishing = prob[1] * 100
    prob_legitimate = prob[0] * 100
    print(f"URL: {url}")
    print(f"Prediction: {label}")
    print(f"Probability (Phishing): {prob_phishing:.2f}%")
    print(f"Probability (Legitimate): {prob_legitimate:.2f}%")
    print("-" * 50)


Predictions for Custom URLs:
--------------------------------------------------
URL: https://www.saffronart.com
Prediction: Phishing
Probability (Phishing): 100.00%
Probability (Legitimate): 0.00%
--------------------------------------------------
URL: https://1.bayarefastrlz.top/
Prediction: Legitimate
Probability (Phishing): 47.00%
Probability (Legitimate): 53.00%
--------------------------------------------------


In [3]:
import qrcode

def generate_qr_from_url(url, output_file="qrcode.png"):
    # Create QR code instance
    qr = qrcode.QRCode(
        version=1,
        error_correction=qrcode.constants.ERROR_CORRECT_L,
        box_size=10,
        border=4,
    )
    
    # Add URL data to QR code
    qr.add_data(url)
    qr.make(fit=True)
    
    # Create image from QR code
    img = qr.make_image(fill_color="black", back_color="white")
    
    # Save the QR code image
    img.save(output_file)
    print(f"QR code saved as {output_file}")

# Example usage
if __name__ == "__main__":
    sample_url = "https://www.rewildingargentina.org/"
    generate_qr_from_url(sample_url, "example_qr.png")

QR code saved as example_qr.png
